# chassis-sim quickstart

Ackermann, differential drive and mecanum chassis behind the **Gymnasium API** — pure NumPy,
CPU-only, no simulator install.

This notebook runs end to end in well under a minute on a free Colab CPU runtime.

In [ ]:
# On a fresh runtime only. Comment out if chassis-sim is already installed.
%pip install -q "chassis-sim[viz]"

In [ ]:
import hashlib
import math

import matplotlib.pyplot as plt
import numpy as np

import chassis_sim as cs

print("chassis-sim", cs.__version__)
print("scenarios", cs.list_scenarios())
print("chassis", cs.available_chassis())

## Three chassis, one API

Each chassis has its own action semantics and steering limits: Ackermann steers with a wheel angle,
differential drive turns by wheel speeds, mecanum can strafe. The observation is a fixed-width float32
vector whose length depends on the chassis (128 for Ackermann and differential drive, 129 for mecanum,
because mecanum commands carry a lateral velocity).

In [ ]:
rng = np.random.default_rng(0)

for name in ("ackermann", "diff_drive", "mecanum"):
    env = cs.make(name)
    obs, info = env.reset(seed=0)
    total = 0.0
    for _ in range(30):
        obs, reward, terminated, truncated, info = env.step(
            rng.uniform(-1.0, 1.0, size=env.action_space.shape)
        )
        total += reward
        if terminated or truncated:
            break
    print(
        f"{name:10s} obs {obs.shape} {obs.dtype}  action {env.action_space.shape}"
        f"  return {total:+7.2f}  bounded_obs {bool(np.all(np.isfinite(env.observation_space.high)))}"
    )
    env.close()

## The observation is nine composable segments

env.obs_spec_describe() reports where every segment lives. Segments can be dropped, reordered or
extended with your own features via cs.ObsSpec, and env.observation_space always matches what the
environment actually emits — including finite bounds per segment.

In [ ]:
env = cs.make("ackermann")
env.reset(seed=0)

for name, sl in env.obs_spec_describe().items():
    print(f"{name:14s} [{sl.start:3d}:{sl.stop:3d})  width {sl.stop - sl.start}")

space = env.observation_space
print()
print("total dim", space.shape[0])
print(
    "all bounds finite",
    bool(np.all(np.isfinite(space.low))),
    bool(np.all(np.isfinite(space.high))),
)
print("low  min", float(space.low.min()), " high max", float(space.high.max()))
print("reset observation inside bounds", bool(space.contains(env.reset(seed=0)[0])))

## Seeded resets, freshly drawn scenes

reset(seed=...) reproduces the same scene; a bare reset() draws a new one, so consecutive episodes
are not replays of a single map. The default scene is dense_static, which needs no planning extra.

The hashes below are of the rendered frame, so they cover the obstacles as well as the robot.

In [ ]:
view = cs.make("ackermann", render_mode="rgb_array")


def frame_hash(seed=None):
    view.reset(seed=seed)
    return hashlib.sha256(view.render().tobytes()).hexdigest()[:12]


same_seed = (frame_hash(0), frame_hash(0))
fresh = (frame_hash(), frame_hash())
print("reset(seed=0) twice:", same_seed, "identical:", same_seed[0] == same_seed[1])
print("reset() twice      :", fresh, "identical:", fresh[0] == fresh[1])

## Render a frame

render() returns a uint8 image with shape (H, W, 3) when the environment is created with
render_mode='rgb_array'; render_mode='human' shows the same picture in a window.

In [ ]:
env = cs.make("ackermann", render_mode="rgb_array")
env.reset(seed=0)
total = 0.0
for k in range(35):
    obs, reward, terminated, truncated, info = env.step(
        np.array([0.25 * math.sin(0.25 * k), 0.6])
    )
    total += reward
    if terminated or truncated:
        break

frame = env.render()
print("frame", frame.shape, frame.dtype, "return", round(total, 2))

plt.figure(figsize=(5, 5), dpi=120)
plt.imshow(frame)
plt.axis("off")
plt.show()

## Score a controller

cs.evaluate runs an episode budget against a controller and reports success, collisions, mean steps and
map independence. For multi-scenario, multi-seed benchmark tables with confidence intervals use
python -m chassis_sim benchmark (see the README).

In [ ]:
env = cs.make("ackermann")

for kind in ("pp", "dwa"):
    result = cs.evaluate(
        env, cs.make_controller(kind, env=env), episodes=3, max_steps=150
    )
    print(
        f"{kind:9s} success {result.success_rate:5.1%}  collisions {result.collision_rate:5.1%}"
        f"  mean steps {result.mean_steps:6.1f}  distinct maps {result.distinct_maps}/{result.n}"
    )

## Where to go next

* Observations and actions: README.md — Observation and action contract
* Custom segments and sensors: examples/sensors_demo.py, examples/scenario_demo.py
* Benchmark protocol: examples/benchmark_baselines.py, python -m chassis_sim benchmark --help
* Rendering and GIF export: examples/render_demo.py